
# YAMNet Gunshot Detection

In [1]:

# ========================
# USER SETTINGS (EDIT ME)
# ========================

from pathlib import Path

# Point this to the RANGE folder, not a single day folder:
# .../01_FR46/05_05_25 to 05_20_25
LAB_ROOT = Path("/Volumes/aid_elephants_interaction/Audio Data/2025/01_FR46/05_05_25 to 05_20_25")

# Limit files scanned per day folder (set None to scan all files per day)
MAX_FILES_PER_DAY = 3

# Detection tuning
THRESHOLD = 0.35          # raise to reduce false positives
MIN_SEPARATION_SEC = 2.0  # minimum spacing between detections within a file
TOP_K_PER_FILE = 5        # number of top timestamps per file

# Streaming settings
BLOCK_SEC = 120.0         # process audio in 120s chunks (fewer mount reads)
TARGET_SR = 16000         # YAMNet expects 16kHz

# Network-mount robustness
USE_LOCAL_CACHE = True
CACHE_DIR = Path("wav_cache")   # local folder on your machine
CACHE_DIR.mkdir(exist_ok=True)

# Output directory
OUT_DIR = Path("gunshot_outputs")
OUT_DIR.mkdir(exist_ok=True)

print("LAB_ROOT:", LAB_ROOT)
print("USE_LOCAL_CACHE:", USE_LOCAL_CACHE)
print("CACHE_DIR:", CACHE_DIR.resolve())
print("OUT_DIR:", OUT_DIR.resolve())


LAB_ROOT: /Volumes/aid_elephants_interaction/Audio Data/2025/01_FR46/05_05_25 to 05_20_25
USE_LOCAL_CACHE: True
CACHE_DIR: /Users/minyoungkim/Desktop/elephant_acoustic_monitoring/wav_cache
OUT_DIR: /Users/minyoungkim/Desktop/elephant_acoustic_monitoring/gunshot_outputs


In [2]:

# ========================
# IMPORTS & MODEL LOADING
# ========================

import time
import shutil
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import librosa
import librosa.display
import soundfile as sf
from tqdm import tqdm
import tensorflow_hub as hub
import tensorflow as tf

yamnet = hub.load("https://tfhub.dev/google/yamnet/1")

# Hard-coded gunshot index for YAMNet v1
GUNSHOT_CLASS_INDEX = 418
print("Using GUNSHOT_CLASS_INDEX =", GUNSHOT_CLASS_INDEX)


/Users/minyoungkim/miniforge3/envs/dsc80/lib/python3.12/site-packages/google/protobuf/runtime_version.py:98: UserWarning: Protobuf gencode version 5.28.3 is exactly one major version older than the runtime version 6.30.2 at tensorflow/core/framework/attr_value.proto. Please update the gencode to avoid compatibility violations in the next runtime release.
  warnings.warn(
/Users/minyoungkim/miniforge3/envs/dsc80/lib/python3.12/site-packages/google/protobuf/runtime_version.py:98: UserWarning: Protobuf gencode version 5.28.3 is exactly one major version older than the runtime version 6.30.2 at tensorflow/core/framework/tensor.proto. Please update the gencode to avoid compatibility violations in the next runtime release.
  warnings.warn(
/Users/minyoungkim/miniforge3/envs/dsc80/lib/python3.12/site-packages/google/protobuf/runtime_version.py:98: UserWarning: Protobuf gencode version 5.28.3 is exactly one major version older than the runtime version 6.30.2 at tensorflow/core/framework/resour

Using GUNSHOT_CLASS_INDEX = 418


In [3]:
def safe_unlink(path: Path):
    try:
        if path.exists():
            path.unlink()
    except Exception as e:
        print(f"[cache] Could not delete {path}: {e}")

In [4]:

# ========================
# DISCOVER DAY FOLDERS
# ========================

def find_day_folders(root: Path):
    """Return subfolders under root that look like YYYYMMDD."""
    root = Path(root)
    if not root.exists():
        raise FileNotFoundError(f"LAB_ROOT does not exist: {root}")
    days = []
    for p in root.iterdir():
        if p.is_dir() and p.name.isdigit() and len(p.name) == 8:
            days.append(p)
    return sorted(days)

def find_wavs_in_day(day_folder: Path):
    """Find wav/WAV files recursively inside a day folder."""
    return sorted([
        p for p in Path(day_folder).rglob("*")
        if p.is_file() and p.suffix.lower() == ".wav"
    ])

day_folders = find_day_folders(LAB_ROOT)
print("Total day folders found:", len(day_folders))
print("First 10 day folders:", [d.name for d in day_folders[:10]])


Total day folders found: 10
First 10 day folders: ['19700101', '20250505', '20250506', '20250507', '20250508', '20250509', '20250510', '20250511', '20250512', '20250513']


In [5]:

# ========================
# LOCAL CACHING (ROBUST)
# ========================

def local_cache_path(src: Path, cache_dir: Path) -> Path:
    # If collisions are possible, change to include parent info
    return cache_dir / src.name

def ensure_local_copy(src: Path, cache_dir: Path, retries=3, sleep_sec=8) -> Path | None:
    """Try to copy src -> cache. If it fails (mount hiccup), return None so we can skip."""
    src = Path(src)
    dst = local_cache_path(src, cache_dir)

    # Already cached and sizes match
    try:
        if dst.exists() and dst.stat().st_size == src.stat().st_size:
            return dst
    except Exception:
        pass

    for attempt in range(1, retries + 1):
        try:
            shutil.copy2(src, dst)
            return dst
        except Exception as e:
            print(f"[cache] Copy failed (attempt {attempt}/{retries}) for {src}: {type(e).__name__}: {e}")
            time.sleep(sleep_sec)

    print(f"[cache] SKIPPING (unreadable right now): {src}")
    return None


In [6]:

# ========================
# STREAMING READ HELPERS
# ========================

def to_mono(x: np.ndarray) -> np.ndarray:
    if x.ndim == 1:
        return x
    return np.mean(x, axis=1)

def resample_to_16k(y: np.ndarray, orig_sr: int, target_sr: int = TARGET_SR) -> np.ndarray:
    if orig_sr == target_sr:
        return y.astype(np.float32)
    return librosa.resample(y.astype(np.float32), orig_sr=orig_sr, target_sr=target_sr)

def yamnet_scores_for_block(y_16k: np.ndarray):
    scores, _, _ = yamnet(y_16k.astype(np.float32))
    return scores.numpy()

def iter_audio_blocks(path: Path, block_sec: float):
    """Yield (block_audio, sr, start_time_sec) from a WAV without loading full file."""
    with sf.SoundFile(str(path)) as f:
        sr = f.samplerate
        channels = f.channels
        block_frames = int(block_sec * sr)

        idx = 0
        while True:
            data = f.read(block_frames, dtype="float32", always_2d=(channels > 1))
            if data is None or len(data) == 0:
                break
            if data.ndim == 2:
                data = to_mono(data)
            start_time = idx * block_sec
            yield data, sr, start_time
            idx += 1


In [7]:

# ========================
# DETECTION: PEAK PICKING
# ========================

def peak_pick_times(times, scores, top_k=5, min_separation_sec=1.0):
    order = np.argsort(scores)[::-1]
    chosen = []
    for i in order:
        t = float(times[i])
        if all(abs(t - ct) >= min_separation_sec for ct in chosen):
            chosen.append(t)
        if len(chosen) >= top_k:
            break
    return chosen


In [8]:

# ========================
# SCAN ONE FILE (STREAMING)
# ========================

def scan_file_streaming(
    wav_path: Path,
    gunshot_class_index: int,
    threshold: float,
    top_k: int,
    min_separation_sec: float,
    block_sec: float,
    use_local_cache: bool,
    cache_dir: Path
):
    src_path = Path(wav_path)

    # Local cache to avoid /Volumes timeouts
    path = src_path
    if use_local_cache:
        cached = ensure_local_copy(src_path, cache_dir)
        if cached is None:
            return pd.DataFrame(columns=["file","time_sec","gunshot_prob"])
        path = cached

    all_probs = []
    all_times = []

    hop_sec = 0.48  # YAMNet frame spacing (approx)

    try:
        for block, sr, block_start in iter_audio_blocks(path, block_sec=block_sec):
            y16 = resample_to_16k(block, orig_sr=sr, target_sr=TARGET_SR)
            scores = yamnet_scores_for_block(y16)
            probs = scores[:, gunshot_class_index]
            times = block_start + (np.arange(len(probs)) * hop_sec)

            all_probs.append(probs)
            all_times.append(times)
    except Exception as e:
        print(f"[scan] SKIPPING due to read/processing error: {src_path} | {type(e).__name__}: {e}")
        return pd.DataFrame(columns=["file","time_sec","gunshot_prob"])

    if len(all_probs) == 0:
        return pd.DataFrame(columns=["file","time_sec","gunshot_prob"])

    probs = np.concatenate(all_probs)
    times = np.concatenate(all_times)

    if np.max(probs) < threshold:
        return pd.DataFrame(columns=["file","time_sec","gunshot_prob"])

    top_times = peak_pick_times(times, probs, top_k=top_k, min_separation_sec=min_separation_sec)

    rows = []
    for t in top_times:
        i = int(np.argmin(np.abs(times - t)))
        if probs[i] >= threshold:
            rows.append({"file": str(src_path), "time_sec": float(times[i]), "gunshot_prob": float(probs[i])})

    return pd.DataFrame(rows).sort_values("gunshot_prob", ascending=False)


In [9]:

# ========================
# SAVE SPECTROGRAM + CLIP
# ========================

def load_snippet(path: Path, center_time_sec: float, segment_sec: float = 3.0):
    with sf.SoundFile(str(path)) as f:
        sr = f.samplerate
        start_frame = max(0, int((center_time_sec - segment_sec/2) * sr))
        n_frames = int(segment_sec * sr)
        f.seek(start_frame)
        data = f.read(n_frames, dtype="float32", always_2d=(f.channels > 1))
        if data.ndim == 2:
            data = np.mean(data, axis=1)
    return data, sr

def save_segment_spectrogram(src_path: Path, center_time_sec: float, out_png: Path, segment_sec: float = 3.0):
    path = ensure_local_copy(src_path, CACHE_DIR) if USE_LOCAL_CACHE else src_path
    if path is None:
        return

    y, sr = load_snippet(path, center_time_sec, segment_sec=segment_sec)
    y16 = resample_to_16k(y, orig_sr=sr, target_sr=TARGET_SR)

    S = librosa.stft(y16, n_fft=1024, hop_length=256)
    S_db = librosa.amplitude_to_db(np.abs(S), ref=np.max)

    plt.figure(figsize=(10, 4))
    librosa.display.specshow(S_db, sr=TARGET_SR, hop_length=256, x_axis="time", y_axis="hz")
    plt.colorbar(format="%+2.0f dB")
    plt.title(f"{src_path.name} | t={center_time_sec:.2f}s")
    plt.tight_layout()
    plt.savefig(out_png, dpi=200)
    plt.close()

def save_audio_clip(src_path: Path, center_time_sec: float, out_wav: Path, segment_sec: float = 3.0):
    path = ensure_local_copy(src_path, CACHE_DIR) if USE_LOCAL_CACHE else src_path
    if path is None:
        return

    y, sr = load_snippet(path, center_time_sec, segment_sec=segment_sec)
    sf.write(str(out_wav), y, sr)


In [10]:

# ========================
# RUN: SCAN ALL DAYS
# ========================

all_hits = []
skipped_days = []
files_seen = 0
files_with_hits = 0

for day in day_folders:
    print(f"\n=== Processing day {day.name} ===")
    wavs = find_wavs_in_day(day)
    print(f"  WAV files found: {len(wavs)}")

    if len(wavs) == 0:
        skipped_days.append(day.name)
        continue

    if MAX_FILES_PER_DAY is not None:
        wavs = wavs[:MAX_FILES_PER_DAY]

    for wav in tqdm(wavs, desc=f"Scanning {day.name}", leave=False):
        files_seen += 1
        hits = scan_file_streaming(
            wav_path=wav,
            gunshot_class_index=GUNSHOT_CLASS_INDEX,
            threshold=THRESHOLD,
            top_k=TOP_K_PER_FILE,
            min_separation_sec=MIN_SEPARATION_SEC,
            block_sec=BLOCK_SEC,
            use_local_cache=USE_LOCAL_CACHE,
            cache_dir=CACHE_DIR
        )

        # free disk space immediately
        cached_path = CACHE_DIR / wav.name
        safe_unlink(cached_path)

        if len(hits) == 0:
            continue

        files_with_hits += 1

        # Save artifacts + add day column
        for r in hits.itertuples(index=False):
            base = Path(r.file).stem
            t = r.time_sec
            spec_path = OUT_DIR / f"{day.name}_{base}_t{t:.2f}_spec.png"
            clip_path = OUT_DIR / f"{day.name}_{base}_t{t:.2f}_clip.wav"
            save_segment_spectrogram(Path(r.file), t, spec_path, segment_sec=3.0)
            save_audio_clip(Path(r.file), t, clip_path, segment_sec=3.0)

        hits = hits.copy()
        hits.insert(0, "day", day.name)
        all_hits.append(hits)

# Combined results
results = (
    pd.concat(all_hits, ignore_index=True)
    if all_hits
    else pd.DataFrame(columns=["day","file","time_sec","gunshot_prob"])
)

print("\nDone.")
print("Days with no WAVs:", skipped_days[:10], f"(total {len(skipped_days)})")
print("Files scanned:", files_seen)
print("Files with >=1 hit:", files_with_hits)
print("Total detections:", len(results))

# Show top results
results.sort_values("gunshot_prob", ascending=False).head(30)



=== Processing day 19700101 ===
  WAV files found: 1



=== Processing day 20250505 ===
  WAV files found: 5



=== Processing day 20250506 ===
  WAV files found: 10



=== Processing day 20250507 ===
  WAV files found: 10



=== Processing day 20250508 ===
  WAV files found: 10


...


=== Processing day 20250511 ===
  WAV files found: 10



=== Processing day 20250512 ===
  WAV files found: 10



=== Processing day 20250513 ===
  WAV files found: 10



Done.
Days with no WAVs: [] (total 0)
Files scanned: 28
Files with >=1 hit: 4
Total detections: 9


,day,file,time_sec,gunshot_prob
0,20250505,/Volumes/aid_elephants_interaction/Audio Data/...,553.92,0.580073
1,20250505,/Volumes/aid_elephants_interaction/Audio Data/...,559.20,0.576523
2,20250505,/Volumes/aid_elephants_interaction/Audio Data/...,640.80,0.513595
3,20250505,/Volumes/aid_elephants_interaction/Audio Data/...,644.64,0.511963
4,20250505,/Volumes/aid_elephants_interaction/Audio Data/...,584.16,0.504703
7,20250512,/Volumes/aid_elephants_interaction/Audio Data/...,3059.04,0.484356
6,20250510,/Volumes/aid_elephants_interaction/Audio Data/...,1479.84,0.398260
5,20250509,/Volumes/aid_elephants_interaction/Audio Data/...,3358.56,0.369355
8,20250512,/Volumes/aid_elephants_interaction/Audio Data/...,543.36,0.350390


In [11]:

# ========================
# OPTIONAL: SAVE CSV SUMMARY
# ========================

results.to_csv("gunshot_summary.csv", index=False)
print("Wrote gunshot_summary.csv")


Wrote gunshot_summary.csv
